#### Core WBT functions

In [ ]:
import os
import geopandas as gpd
from shapely.geometry import LineString
from whitebox import WhiteboxTools
import tempfile

# Initialize WBT
wbt = WhiteboxTools()
workdir = tempfile.mkdtemp()
wbt.set_working_dir(workdir)

# --- Inputs ---
basin_shp = r"C:\TC_Calculator_Work\Basin_BW.shp"
dem_tif = r"C:\TC_Calculator_Work\BW_SURF.tif"
stream_shp = r"C:\TC_Calculator_Work\streams.shp"
output_shp = r"C:\TC_Calculator_Work\TC_Basin_WBT.shp"

# --- DEM Conditioning ---
# Burn streams into DEM
burned_dem = os.path.join(workdir, "dem_burned.tif")
wbt.burn_streams_at_roads(
    dem=dem_tif,
    streams=stream_shp,
    output=burned_dem,
    width=10  # adjust burn-in width in map units
)

# Fill depressions / breach pits
filled_dem = os.path.join(workdir, "dem_filled.tif")
wbt.fill_depressions(dem=burned_dem, output=filled_dem)

# Flow direction (D8)
fdir = os.path.join(workdir, "flowdir.tif")
wbt.d8_pointer(dem=filled_dem, output=fdir)

# Flow accumulation
facc = os.path.join(workdir, "flowacc.tif")
wbt.d8_flow_accumulation(i=fdir, output=facc, out_type="cells")

# Longest flow path within each basin
# (WBT has a tool for this)
lflow = os.path.join(workdir, "longest_flowpath.tif")
wbt.longest_flowpath(dem=filled_dem, output=lflow)

# Convert raster flow path to vector
flowline = os.path.join(workdir, "flowline.shp")
wbt.raster_streams_to_vector(streams=lflow, d8_pntr=fdir, output=flowline)

# --- Post-processing with GeoPandas ---
basins_gdf = gpd.read_file(basin_shp)
flow_gdf = gpd.read_file(flowline)

# Spatial join: assign flow paths to basins
joined = gpd.sjoin(flow_gdf, basins_gdf, predicate="intersects")

# Compute lengths & slopes
joined["length_ft"] = joined.geometry.length * 3.28084  # if DEM units are meters
# slope can be derived from z difference / length
# (would need DEM sample along line or endpoints)

# Classify into sheet vs shallow concentrated (first 100 ft vs remainder)
split_records = []
for _, row in joined.iterrows():
    line = row.geometry
    if line.length > 0:
        if line.length <= 30.48:  # ~100 ft
            split_records.append({**row, "type": "sheet"})
        else:
            sheet = line.interpolate(30.48)
            split_records.append({**row, "geometry": LineString([line.coords[0], sheet]), "type": "sheet"})
            split_records.append({**row, "geometry": LineString([sheet, line.coords[-1]]), "type": "shallow"})
            
split_gdf = gpd.GeoDataFrame(split_records, crs=joined.crs)
split_gdf.to_file(output_shp)

print(f"Saved Tc flow paths to {output_shp}")


### Refactor notes (what changed vs. original script)

- Hydro-enforcing & conditioning: `fill_depressions`, `resolve_flats`, and pit/depression handling are replaced by WBT’s `fill_depressions` (and optional `breach_depressions`), with optional Gaussian smoothing to stabilize flats (enhancement of original `gaussian_filter`)
- Stream burning: if input streams are vectors, we rasterize them, then use `burn_streams` (WBT) instead of the original manual loop that subtracted a burn depth from DEM cells
- Flow direction / accumulation: `d8_pointer` + `d8_flow_accumulation` replace `grid.flowdir` / `grid.accumulation`
- Longest flow path & pour point selection: computes a per-basin outlet at the max accumulation along the basin boundary (original approach), then use WBT’s `flow_length` (upslope) to find the farthest cell and trace a vector flowpath with `trace_downslope_flowpaths`.
- Slope along path: original `calculate_slope` has been ported by sampling the DEM along the traced polyline.
- Sheet vs shallow: splits the first 100 ft from the remainder, as original implementaion did with substring (now done with Shapely line segmentation)
- Environment: Designed to run cleanly in Jupyter (WBT logs progress inline) without pysheds dependency.

#### Imports

In [ ]:
import os, math, tempfile, gc
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString, Point
from shapely.ops import substring
import rasterio
from rasterio import features
from rasterio.mask import mask as rio_mask
from whitebox import WhiteboxTools

#### Inputs and Parameters


In [ ]:
dem_tif = r"C:\TC_Calculator_Work\BW_SURF.tif"
basin_shp = r"C:\TC_Calculator_Work\Basin_BW.shp"        # polygons (one or many)
stream_shp_list = [
    r"C:\TC_Calculator_Work\NHD_H_03080102_HU8_Shape\Shape\NHDFlowline.shp",
    r"C:\TC_Calculator_Work\NHD_H_03080101_HU8_Shape\Shape\NHDFlowline.shp"
]
output_tc_lines_shp = r"C:\TC_Calculator_Work\TC_Basin_WBT.shp"

In [ ]:
SIGMA_VALUES_TO_TEST = [2.0]      # smoothing sigma(s) for flats (DEM units)
MINIMUM_PATH_LENGTH_FT = 100.0    # minimum total path length to accept
BUFFER_FT = 75.0                  # clip buffer around basins
BURN_DEPTH = 1.5                  # depth used in burning (map units of DEM)
SHEET_DIST_FT = 100.0             # length for sheet-flow segment
CHANNEL_THRESHOLD_CELLS = 1500    # used for stream extraction (optional)

#### Initial Setup

In [ ]:
# ---- Setup WBT ----
wbt = WhiteboxTools()
workdir = tempfile.mkdtemp()
wbt.set_working_dir(workdir)

print("Working dir:", workdir)

# ---- Load base layers ----
basins_gdf = gpd.read_file(basin_shp)
streams = [gpd.read_file(s) for s in stream_shp_list]
streams_gdf = pd.concat(streams, ignore_index=True)

# ---- Ensure all layers share a CRS with DEM ----
with rasterio.open(dem_tif) as src:
    dem_crs = src.crs
    dem_transform = src.transform
    dem_resx = src.res[0]
    dem_resy = src.res[1]
    dem_units_m = "meter" in (src.crs.axis_info[0].unit_name.lower()
                              if src.crs and src.crs.axis_info else "meter")

if basins_gdf.crs != dem_crs:
    basins_gdf = basins_gdf.to_crs(dem_crs)
if streams_gdf.crs != dem_crs:
    streams_gdf = streams_gdf.to_crs(dem_crs)

#### Process DEM

In [ ]:
# ---- Compute buffered bounds and clip DEM once ----
xmin, ymin, xmax, ymax = basins_gdf.total_bounds
# convert buffer ft->map units if DEM is meters
ft_to_map = 0.3048 if dem_units_m else 1.0
buf = BUFFER_FT * ft_to_map
bounds = (xmin - buf, ymin - buf, xmax + buf, ymax + buf)

with rasterio.open(dem_tif) as src:
    window = rasterio.windows.from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
    dem_clip = src.read(1, window=window).astype("float32")
    meta = src.profile.copy()
    meta.update({
        'height': dem_clip.shape[0],
        'width': dem_clip.shape[1],
        'transform': rasterio.windows.transform(window, src.transform),
        'dtype': 'float32'
    })

# Save clipped DEM for WBT processing
dem_clip_path = os.path.join(workdir, "dem_clip.tif")
with rasterio.open(dem_clip_path, "w", **meta) as dst:
    dst.write(dem_clip, 1)

#### Rasterize Streams

In [ ]:
# ---- Rasterize streams to DEM grid, then burn ----
stream_ras = os.path.join(workdir, "streams_ras.tif")
# value=1 where streams exist
shapes = [(geom, 1) for geom in streams_gdf.geometry if geom and not geom.is_empty]
out_streams = np.zeros_like(dem_clip, dtype="uint8")
if shapes:
    out_streams = features.rasterize(
        shapes=shapes,
        out_shape=dem_clip.shape,
        transform=meta['transform'],
        fill=0,
        dtype="uint8"
    )

with rasterio.open(stream_ras, "w", driver="GTiff",
                   height=out_streams.shape[0], width=out_streams.shape[1],
                   count=1, dtype="uint8", crs=dem_crs, transform=meta['transform']) as dst:
    dst.write(out_streams, 1)

burned_dem = os.path.join(workdir, "dem_burned.tif")
# WBT burn_streams requires a streams raster coincident with DEM
wbt.burn_streams(dem=dem_clip_path, streams=stream_ras, output=burned_dem, burn= BURN_DEPTH)

final_outputs = []  # collect best lines per basin across sigma runs
basin_ids = basins_gdf.index.tolist()

for sigma in SIGMA_VALUES_TO_TEST:
    print(f"\n=== Running with sigma = {sigma} ===")

    # Optional smoothing for flat areas (WBT gaussian filter)
    smoothed_dem = os.path.join(workdir, f"dem_smoothed_sigma{sigma}.tif")
    if sigma and sigma > 0:
        wbt.gaussian_filter(i=burned_dem, output=smoothed_dem, sigma=sigma)
        dem_for_flow = smoothed_dem
    else:
        dem_for_flow = burned_dem

    # Fill depressions (and optionally breach if your area is very flat)
    filled_dem = os.path.join(workdir, f"dem_filled_sigma{sigma}.tif")
    wbt.fill_depressions(dem=dem_for_flow, output=filled_dem)
    # wbt.breach_depressions(dem=filled_dem, output=filled_dem)  # uncomment if needed

    # Resolve flats is implicit in WBT’s pointer algorithm; it handles flats robustly.
    # Flow direction (D8 pointer) & accumulation
    fdir = os.path.join(workdir, f"fdir_sigma{sigma}.tif")
    facc = os.path.join(workdir, f"facc_sigma{sigma}.tif")
    wbt.d8_pointer(dem=filled_dem, output=fdir)
    wbt.d8_flow_accumulation(i=fdir, output=facc, out_type="cells")

    # Rasterize basins to DEM grid for per-basin ops
    basin_ras = os.path.join(workdir, f"basins_sigma{sigma}.tif")
    basin_idx = np.zeros_like(dem_clip, dtype="int32")
    for i, geom in enumerate(basins_gdf.geometry):
        if geom is None or geom.is_empty:
            continue
        basin_idx = basin_idx + features.rasterize(
            [(geom, i+1)],
            out_shape=dem_clip.shape,
            transform=meta['transform'],
            fill=0,
            dtype="int32"
        )

    with rasterio.open(basin_ras, "w", driver="GTiff",
                       height=basin_idx.shape[0], width=basin_idx.shape[1],
                       count=1, dtype="int32", crs=dem_crs, transform=meta['transform']) as dst:
        dst.write(basin_idx, 1)

    # Load accumulation array for boundary-based outlet selection
    with rasterio.open(facc) as acc_src:
        acc_arr = acc_src.read(1)
    with rasterio.open(basin_ras) as br:
        basin_arr = br.read(1)

    # Utility: get boundary mask for a basin id
    from scipy.ndimage import binary_erosion

    def basin_boundary_mask(bid):
        mask = (basin_arr == bid).astype("uint8")
        if mask.max() == 0:
            return None
        eroded = binary_erosion(mask, border_value=0)
        boundary = (mask - eroded).astype("uint8")
        return boundary

    # For each basin: find outlet @ boundary cell with max accumulation
    # then find farthest upstream cell (max upslope flow length) and trace path
    for bid in range(1, len(basin_ids)+1):
        boundary = basin_boundary_mask(bid)
        if boundary is None or boundary.sum() == 0:
            continue

        # Max accumulation along boundary → outlet
        acc_boundary = np.where(boundary == 1, acc_arr, 0)
        if acc_boundary.max() == 0:
            continue
        outlet_rc = np.unravel_index(np.argmax(acc_boundary), acc_boundary.shape)
        outlet_r, outlet_c = int(outlet_rc[0]), int(outlet_rc[1])

        # Create pour point raster for this basin outlet
        outlet_ras = os.path.join(workdir, f"outlet_bid{bid}_sigma{sigma}.tif")
        outlet_arr = np.zeros_like(acc_arr, dtype="uint8")
        outlet_arr[outlet_r, outlet_c] = 1
        with rasterio.open(outlet_ras, "w", driver="GTiff",
                           height=outlet_arr.shape[0], width=outlet_arr.shape[1],
                           count=1, dtype="uint8", crs=dem_crs, transform=meta['transform']) as dst:
            dst.write(outlet_arr, 1)

        # Upslope flow length to outlet (i.e., distance-to-outlet along flowpaths)
        # WBT's flow_length with "upstream" option
        up_len = os.path.join(workdir, f"up_len_bid{bid}_sigma{sigma}.tif")
        wbt.flow_length(d8_pntr=fdir, output=up_len, direction="upstream", weights=None, outlets=outlet_ras)

        # Within basin, find the cell with **max** upstream flow length → hydraulic start
        with rasterio.open(up_len) as ul:
            ul_arr = ul.read(1)
        # mask by basin id
        ul_arr_masked = np.where(basin_arr == bid, ul_arr, np.nan)
        if np.all(np.isnan(ul_arr_masked)):
            continue
        start_rc = np.unravel_index(np.nanargmax(ul_arr_masked), ul_arr_masked.shape)
        start_r, start_c = int(start_rc[0]), int(start_rc[1])

        # Trace downslope flowpath from start to outlet using the D8 pointer
        starts_ras = os.path.join(workdir, f"starts_bid{bid}_sigma{sigma}.tif")
        starts_arr = np.zeros_like(acc_arr, dtype="uint8")
        starts_arr[start_r, start_c] = 1
        with rasterio.open(starts_ras, "w", driver="GTiff",
                           height=starts_arr.shape[0], width=starts_arr.shape[1],
                           count=1, dtype="uint8", crs=dem_crs, transform=meta['transform']) as dst:
            dst.write(starts_arr, 1)

        traced_ras = os.path.join(workdir, f"trace_bid{bid}_sigma{sigma}.tif")
        wbt.trace_downslope_flowpaths(d8_pntr=fdir, pour_pts=starts_ras, output=traced_ras)

        # Convert path raster to vector line
        traced_vec = os.path.join(workdir, f"trace_bid{bid}_sigma{sigma}.shp")
        wbt.raster_streams_to_vector(streams=traced_ras, d8_pntr=fdir, output=traced_vec)

        # Load vector, pick the longest single-part line within basin
        if not os.path.exists(traced_vec):
            continue
        line_gdf = gpd.read_file(traced_vec)
        if line_gdf.empty:
            continue

        # Intersect with basin polygon to clip line
        basin_poly = basins_gdf.iloc[bid-1].geometry
        line_gdf = gpd.overlay(line_gdf, gpd.GeoDataFrame(geometry=[basin_poly], crs=basins_gdf.crs), how="intersection")
        if line_gdf.empty:
            continue

        # pick longest
        line_gdf["len_mapunits"] = line_gdf.geometry.length
        best = line_gdf.iloc[line_gdf["len_mapunits"].idxmax()]
        best_line = best.geometry

        # compute total length in ft
        map_to_ft = (3.28084 if dem_units_m else 1.0)
        total_len_ft = best_line.length * map_to_ft

        if total_len_ft < MINIMUM_PATH_LENGTH_FT:
            # skip this basin for this sigma
            continue

        # sample elevations along the line to compute slope (simple end-to-end)
        def line_end_elevations(line_geom, dem_path):
            # sample start and end
            start_pt = Point(line_geom.coords[0])
            end_pt   = Point(line_geom.coords[-1])
            coords = [(start_pt.x, start_pt.y), (end_pt.x, end_pt.y)]
            with rasterio.open(dem_path) as src:
                samples = list(src.sample(coords))
                z1, z2 = float(samples[0][0]), float(samples[1][0])
                # handle NODATA if present
                z1 = np.nan if z1 == src.nodata else z1
                z2 = np.nan if z2 == src.nodata else z2
            return z1, z2

        z1, z2 = line_end_elevations(best_line, filled_dem)
        elev_drop = (z1 - z2) if (z1 is not None and z2 is not None and not np.isnan(z1) and not np.isnan(z2)) else np.nan
        slope_ftft = (elev_drop * (3.28084 if dem_units_m else 1.0)) / total_len_ft if total_len_ft > 0 and not np.isnan(elev_drop) else 0.0

        # Split into sheet vs shallow concentrated
        sheet_len_map = SHEET_DIST_FT / map_to_ft
        if best_line.length <= sheet_len_map:
            parts = [dict(
                basin_id=bid,
                type="sheet_flow",
                sigma=sigma,
                slope=slope_ftft,
                length_ft=total_len_ft,
                geometry=best_line
            )]
        else:
            # split at sheet distance
            # substring expects distances in the same units as geometry (map units)
            sheet_seg = substring(best_line, 0, sheet_len_map)
            conc_seg  = substring(best_line, sheet_len_map, best_line.length)
            parts = [
                dict(basin_id=bid, type="sheet_flow",    sigma=sigma, slope=slope_ftft,
                     length_ft=sheet_seg.length * map_to_ft, geometry=sheet_seg),
                dict(basin_id=bid, type="shallow_conc", sigma=sigma, slope=slope_ftft,
                     length_ft=conc_seg.length * map_to_ft,  geometry=conc_seg)
            ]

        final_outputs.extend(parts)

    gc.collect()

#### Assemble and Write Final

In [ ]:
if final_outputs:
    out_gdf = gpd.GeoDataFrame(final_outputs, crs=basins_gdf.crs)
    # Keep only the longest total path per basin across sigma runs
    # Re-aggregate by basin_id: sum sheet+shallow lengths for each sigma, pick max
    totals = (out_gdf.groupby(["basin_id","sigma"])["length_ft"].sum()
                        .reset_index()
                        .sort_values(["basin_id","length_ft"], ascending=[True, False]))
    keep = totals.drop_duplicates(subset=["basin_id"], keep="first")
    keep_pairs = set(map(tuple, keep[["basin_id","sigma"]].values.tolist()))
    out_gdf = out_gdf[[ (r.basin_id, r.sigma) in keep_pairs for _, r in out_gdf.iterrows() ]]

    # Optional: order sheet first, then shallow
    type_order = {"sheet_flow": 0, "shallow_conc": 1}
    out_gdf["type_ord"] = out_gdf["type"].map(type_order)
    out_gdf = out_gdf.sort_values(["basin_id","type_ord"]).drop(columns="type_ord")

    out_gdf.to_file(output_tc_lines_shp)
    print(f"Saved TC flow path segments to: {output_tc_lines_shp}")
else:
    print("No qualifying flow paths were generated.")

#### Single Cell Refactored

In [ ]:
# --- Core imports ---
import os, math, tempfile, gc
import geopandas as gpd
import pandas as pd
import numpy as np
from shapely.geometry import LineString, Point
from shapely.ops import substring
import rasterio
from rasterio import features
from rasterio.mask import mask as rio_mask
from whitebox import WhiteboxTools

# -------------------------
# Inputs (EDIT THESE)
# -------------------------
dem_tif = r"C:\TC_Calculator_Work\BW_SURF.tif"
basin_shp = r"C:\TC_Calculator_Work\Basin_BW.shp"        # polygons (one or many)
stream_shp_list = [
    r"C:\TC_Calculator_Work\NHD_H_03080102_HU8_Shape\Shape\NHDFlowline.shp",
    r"C:\TC_Calculator_Work\NHD_H_03080101_HU8_Shape\Shape\NHDFlowline.shp"
]
output_tc_lines_shp = r"C:\TC_Calculator_Work\TC_Basin_WBT.shp"

# -------------------------
# Parameters (EDIT AS NEEDED)
# -------------------------
SIGMA_VALUES_TO_TEST = [2.0]      # smoothing sigma(s) for flats (DEM units)
MINIMUM_PATH_LENGTH_FT = 100.0    # minimum total path length to accept
BUFFER_FT = 75.0                  # clip buffer around basins
BURN_DEPTH = 1.5                  # depth used in burning (map units of DEM)
SHEET_DIST_FT = 100.0             # length for sheet-flow segment
CHANNEL_THRESHOLD_CELLS = 1500    # used for stream extraction (optional)
# -------------------------

# ---- Setup WBT ----
wbt = WhiteboxTools()
workdir = tempfile.mkdtemp()
wbt.set_working_dir(workdir)

print("Working dir:", workdir)

# ---- Load base layers ----
basins_gdf = gpd.read_file(basin_shp)
streams = [gpd.read_file(s) for s in stream_shp_list]
streams_gdf = pd.concat(streams, ignore_index=True)

# ---- Ensure all layers share a CRS with DEM ----
with rasterio.open(dem_tif) as src:
    dem_crs = src.crs
    dem_transform = src.transform
    dem_resx = src.res[0]
    dem_resy = src.res[1]
    dem_units_m = "meter" in (src.crs.axis_info[0].unit_name.lower()
                              if src.crs and src.crs.axis_info else "meter")

if basins_gdf.crs != dem_crs:
    basins_gdf = basins_gdf.to_crs(dem_crs)
if streams_gdf.crs != dem_crs:
    streams_gdf = streams_gdf.to_crs(dem_crs)

# ---- Compute buffered bounds and clip DEM once ----
xmin, ymin, xmax, ymax = basins_gdf.total_bounds
# convert buffer ft->map units if DEM is meters
ft_to_map = 0.3048 if dem_units_m else 1.0
buf = BUFFER_FT * ft_to_map
bounds = (xmin - buf, ymin - buf, xmax + buf, ymax + buf)

with rasterio.open(dem_tif) as src:
    window = rasterio.windows.from_bounds(*bounds, transform=src.transform).round_offsets().round_lengths()
    dem_clip = src.read(1, window=window).astype("float32")
    meta = src.profile.copy()
    meta.update({
        'height': dem_clip.shape[0],
        'width': dem_clip.shape[1],
        'transform': rasterio.windows.transform(window, src.transform),
        'dtype': 'float32'
    })

# Save clipped DEM for WBT processing
dem_clip_path = os.path.join(workdir, "dem_clip.tif")
with rasterio.open(dem_clip_path, "w", **meta) as dst:
    dst.write(dem_clip, 1)

# ---- Rasterize streams to DEM grid, then burn ----
stream_ras = os.path.join(workdir, "streams_ras.tif")
# value=1 where streams exist
shapes = [(geom, 1) for geom in streams_gdf.geometry if geom and not geom.is_empty]
out_streams = np.zeros_like(dem_clip, dtype="uint8")
if shapes:
    out_streams = features.rasterize(
        shapes=shapes,
        out_shape=dem_clip.shape,
        transform=meta['transform'],
        fill=0,
        dtype="uint8"
    )

with rasterio.open(stream_ras, "w", driver="GTiff",
                   height=out_streams.shape[0], width=out_streams.shape[1],
                   count=1, dtype="uint8", crs=dem_crs, transform=meta['transform']) as dst:
    dst.write(out_streams, 1)

burned_dem = os.path.join(workdir, "dem_burned.tif")
# WBT burn_streams requires a streams raster coincident with DEM
wbt.burn_streams(dem=dem_clip_path, streams=stream_ras, output=burned_dem, burn= BURN_DEPTH)

final_outputs = []  # collect best lines per basin across sigma runs
basin_ids = basins_gdf.index.tolist()

for sigma in SIGMA_VALUES_TO_TEST:
    print(f"\n=== Running with sigma = {sigma} ===")

    # Optional smoothing for flat areas (WBT gaussian filter)
    smoothed_dem = os.path.join(workdir, f"dem_smoothed_sigma{sigma}.tif")
    if sigma and sigma > 0:
        wbt.gaussian_filter(i=burned_dem, output=smoothed_dem, sigma=sigma)
        dem_for_flow = smoothed_dem
    else:
        dem_for_flow = burned_dem

    # Fill depressions (and optionally breach if your area is very flat)
    filled_dem = os.path.join(workdir, f"dem_filled_sigma{sigma}.tif")
    wbt.fill_depressions(dem=dem_for_flow, output=filled_dem)
    # wbt.breach_depressions(dem=filled_dem, output=filled_dem)  # uncomment if needed

    # Resolve flats is implicit in WBT’s pointer algorithm; it handles flats robustly.
    # Flow direction (D8 pointer) & accumulation
    fdir = os.path.join(workdir, f"fdir_sigma{sigma}.tif")
    facc = os.path.join(workdir, f"facc_sigma{sigma}.tif")
    wbt.d8_pointer(dem=filled_dem, output=fdir)
    wbt.d8_flow_accumulation(i=fdir, output=facc, out_type="cells")

    # Rasterize basins to DEM grid for per-basin ops
    basin_ras = os.path.join(workdir, f"basins_sigma{sigma}.tif")
    basin_idx = np.zeros_like(dem_clip, dtype="int32")
    for i, geom in enumerate(basins_gdf.geometry):
        if geom is None or geom.is_empty:
            continue
        basin_idx = basin_idx + features.rasterize(
            [(geom, i+1)],
            out_shape=dem_clip.shape,
            transform=meta['transform'],
            fill=0,
            dtype="int32"
        )

    with rasterio.open(basin_ras, "w", driver="GTiff",
                       height=basin_idx.shape[0], width=basin_idx.shape[1],
                       count=1, dtype="int32", crs=dem_crs, transform=meta['transform']) as dst:
        dst.write(basin_idx, 1)

    # Load accumulation array for boundary-based outlet selection
    with rasterio.open(facc) as acc_src:
        acc_arr = acc_src.read(1)
    with rasterio.open(basin_ras) as br:
        basin_arr = br.read(1)

    # Utility: get boundary mask for a basin id
    from scipy.ndimage import binary_erosion

    def basin_boundary_mask(bid):
        mask = (basin_arr == bid).astype("uint8")
        if mask.max() == 0:
            return None
        eroded = binary_erosion(mask, border_value=0)
        boundary = (mask - eroded).astype("uint8")
        return boundary

    # For each basin: find outlet @ boundary cell with max accumulation
    # then find farthest upstream cell (max upslope flow length) and trace path
    for bid in range(1, len(basin_ids)+1):
        boundary = basin_boundary_mask(bid)
        if boundary is None or boundary.sum() == 0:
            continue

        # Max accumulation along boundary → outlet
        acc_boundary = np.where(boundary == 1, acc_arr, 0)
        if acc_boundary.max() == 0:
            continue
        outlet_rc = np.unravel_index(np.argmax(acc_boundary), acc_boundary.shape)
        outlet_r, outlet_c = int(outlet_rc[0]), int(outlet_rc[1])

        # Create pour point raster for this basin outlet
        outlet_ras = os.path.join(workdir, f"outlet_bid{bid}_sigma{sigma}.tif")
        outlet_arr = np.zeros_like(acc_arr, dtype="uint8")
        outlet_arr[outlet_r, outlet_c] = 1
        with rasterio.open(outlet_ras, "w", driver="GTiff",
                           height=outlet_arr.shape[0], width=outlet_arr.shape[1],
                           count=1, dtype="uint8", crs=dem_crs, transform=meta['transform']) as dst:
            dst.write(outlet_arr, 1)

        # Upslope flow length to outlet (i.e., distance-to-outlet along flowpaths)
        # WBT's flow_length with "upstream" option
        up_len = os.path.join(workdir, f"up_len_bid{bid}_sigma{sigma}.tif")
        wbt.flow_length(d8_pntr=fdir, output=up_len, direction="upstream", weights=None, outlets=outlet_ras)

        # Within basin, find the cell with **max** upstream flow length → hydraulic start
        with rasterio.open(up_len) as ul:
            ul_arr = ul.read(1)
        # mask by basin id
        ul_arr_masked = np.where(basin_arr == bid, ul_arr, np.nan)
        if np.all(np.isnan(ul_arr_masked)):
            continue
        start_rc = np.unravel_index(np.nanargmax(ul_arr_masked), ul_arr_masked.shape)
        start_r, start_c = int(start_rc[0]), int(start_rc[1])

        # Trace downslope flowpath from start to outlet using the D8 pointer
        starts_ras = os.path.join(workdir, f"starts_bid{bid}_sigma{sigma}.tif")
        starts_arr = np.zeros_like(acc_arr, dtype="uint8")
        starts_arr[start_r, start_c] = 1
        with rasterio.open(starts_ras, "w", driver="GTiff",
                           height=starts_arr.shape[0], width=starts_arr.shape[1],
                           count=1, dtype="uint8", crs=dem_crs, transform=meta['transform']) as dst:
            dst.write(starts_arr, 1)

        traced_ras = os.path.join(workdir, f"trace_bid{bid}_sigma{sigma}.tif")
        wbt.trace_downslope_flowpaths(d8_pntr=fdir, pour_pts=starts_ras, output=traced_ras)

        # Convert path raster to vector line
        traced_vec = os.path.join(workdir, f"trace_bid{bid}_sigma{sigma}.shp")
        wbt.raster_streams_to_vector(streams=traced_ras, d8_pntr=fdir, output=traced_vec)

        # Load vector, pick the longest single-part line within basin
        if not os.path.exists(traced_vec):
            continue
        line_gdf = gpd.read_file(traced_vec)
        if line_gdf.empty:
            continue

        # Intersect with basin polygon to clip line
        basin_poly = basins_gdf.iloc[bid-1].geometry
        line_gdf = gpd.overlay(line_gdf, gpd.GeoDataFrame(geometry=[basin_poly], crs=basins_gdf.crs), how="intersection")
        if line_gdf.empty:
            continue

        # pick longest
        line_gdf["len_mapunits"] = line_gdf.geometry.length
        best = line_gdf.iloc[line_gdf["len_mapunits"].idxmax()]
        best_line = best.geometry

        # compute total length in ft
        map_to_ft = (3.28084 if dem_units_m else 1.0)
        total_len_ft = best_line.length * map_to_ft

        if total_len_ft < MINIMUM_PATH_LENGTH_FT:
            # skip this basin for this sigma
            continue

        # sample elevations along the line to compute slope (simple end-to-end)
        def line_end_elevations(line_geom, dem_path):
            # sample start and end
            start_pt = Point(line_geom.coords[0])
            end_pt   = Point(line_geom.coords[-1])
            coords = [(start_pt.x, start_pt.y), (end_pt.x, end_pt.y)]
            with rasterio.open(dem_path) as src:
                samples = list(src.sample(coords))
                z1, z2 = float(samples[0][0]), float(samples[1][0])
                # handle NODATA if present
                z1 = np.nan if z1 == src.nodata else z1
                z2 = np.nan if z2 == src.nodata else z2
            return z1, z2

        z1, z2 = line_end_elevations(best_line, filled_dem)
        elev_drop = (z1 - z2) if (z1 is not None and z2 is not None and not np.isnan(z1) and not np.isnan(z2)) else np.nan
        slope_ftft = (elev_drop * (3.28084 if dem_units_m else 1.0)) / total_len_ft if total_len_ft > 0 and not np.isnan(elev_drop) else 0.0

        # Split into sheet vs shallow concentrated
        sheet_len_map = SHEET_DIST_FT / map_to_ft
        if best_line.length <= sheet_len_map:
            parts = [dict(
                basin_id=bid,
                type="sheet_flow",
                sigma=sigma,
                slope=slope_ftft,
                length_ft=total_len_ft,
                geometry=best_line
            )]
        else:
            # split at sheet distance
            # substring expects distances in the same units as geometry (map units)
            sheet_seg = substring(best_line, 0, sheet_len_map)
            conc_seg  = substring(best_line, sheet_len_map, best_line.length)
            parts = [
                dict(basin_id=bid, type="sheet_flow",    sigma=sigma, slope=slope_ftft,
                     length_ft=sheet_seg.length * map_to_ft, geometry=sheet_seg),
                dict(basin_id=bid, type="shallow_conc", sigma=sigma, slope=slope_ftft,
                     length_ft=conc_seg.length * map_to_ft,  geometry=conc_seg)
            ]

        final_outputs.extend(parts)

    gc.collect()

# ---- Assemble & write final output ----
if final_outputs:
    out_gdf = gpd.GeoDataFrame(final_outputs, crs=basins_gdf.crs)
    # Keep only the longest total path per basin across sigma runs
    # Re-aggregate by basin_id: sum sheet+shallow lengths for each sigma, pick max
    totals = (out_gdf.groupby(["basin_id","sigma"])["length_ft"].sum()
                        .reset_index()
                        .sort_values(["basin_id","length_ft"], ascending=[True, False]))
    keep = totals.drop_duplicates(subset=["basin_id"], keep="first")
    keep_pairs = set(map(tuple, keep[["basin_id","sigma"]].values.tolist()))
    out_gdf = out_gdf[[ (r.basin_id, r.sigma) in keep_pairs for _, r in out_gdf.iterrows() ]]

    # Optional: order sheet first, then shallow
    type_order = {"sheet_flow": 0, "shallow_conc": 1}
    out_gdf["type_ord"] = out_gdf["type"].map(type_order)
    out_gdf = out_gdf.sort_values(["basin_id","type_ord"]).drop(columns="type_ord")

    out_gdf.to_file(output_tc_lines_shp)
    print(f"Saved TC flow path segments to: {output_tc_lines_shp}")
else:
    print("No qualifying flow paths were generated.")
